# Figure 2 regeneration — real λ2=1.0 attention checkpoint

Regenerates the qualitative attention-drift comparison figure using the actual λ2=1.0 attention-consistency winner (`l2_1_mse`) instead of the stale λ2=0.3 checkpoint the paper's caption has been flagging as "map pending". Only runs 3 forward passes + Grad-Rollout — a GPU is not required, CPU is fine and should take well under a minute.

Upload **only** `figure2_colab.zip` (from `make_figure2_colab_zip.py`), not the whole repo.

**Before running:** upload the zip to `MyDrive/figure2_colab.zip`. You will also be asked partway through to upload `segformer_b0_vanilla_best.pt` (Kalana's full-scale vanilla SegFormer-B0 checkpoint) — it isn't bundled since it only ever lived on Drive, never got fetched to the machine that built this zip. The λ2=1.0 attention checkpoint (`segformer_b0_att_best.pt`) IS already bundled — no need to re-upload it.

## Step 0: Unzip + deps

In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/figure2_colab.zip")
BUNDLE = Path("/content/figure2")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    if not (HERE / "paths.py").exists():
        cand = Path("Phase2/Dhinanjaya-Person5/figure2").resolve()
        if (cand / "paths.py").exists():
            HERE = cand

sys.path.insert(0, str(HERE))
print("HERE =", HERE)

In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")

## Step 1: Upload the vanilla checkpoint (if missing)

In [ ]:
import paths

vanilla_ckpt = paths.CKPT_DIR / "segformer_b0_vanilla_best.pt"
att_ckpt = paths.CKPT_DIR / "segformer_b0_att_best.pt"
print("CKPT_DIR:", paths.CKPT_DIR)
print("att checkpoint present:", att_ckpt.exists())
print("vanilla checkpoint present:", vanilla_ckpt.exists())

assert att_ckpt.exists(), "segformer_b0_att_best.pt missing -- rebuild the zip with make_figure2_colab_zip.py"

if not vanilla_ckpt.exists():
    if IN_COLAB:
        print("Upload segformer_b0_vanilla_best.pt (Kalana's full-scale vanilla checkpoint):")
        from google.colab import files
        uploaded = files.upload()
        (name,) = uploaded.keys()
        shutil.move(name, str(vanilla_ckpt))
    else:
        raise FileNotFoundError(
            f"Copy Kalana's segformer_b0_vanilla_best.pt to {vanilla_ckpt} and re-run this cell."
        )

assert vanilla_ckpt.exists()
print("Both checkpoints ready.")

## Step 2: Device check (CPU is fine for 3 images)

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Step 3: Generate the figures

Default args (`--n 3 --seed 123`) reproduce the same 3 held-out demo images every existing attention-drift figure in this repo was generated from — keeps "not otherwise cherry-picked" true.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "generate_full_scale_figures.py"], cwd=str(HERE))

## Step 4: Preview + download

In [ ]:
import paths
from IPython.display import Image, display

out_dir = paths.RESULTS_DIR / "attention_drift_figures"
pngs = sorted(out_dir.glob("attention_drift_*_full_scale.png"))
print(f"{len(pngs)} figure(s) in {out_dir}")
for p in pngs:
    print(p.name)
    display(Image(filename=str(p)))

if IN_COLAB:
    from google.colab import files
    for p in pngs:
        files.download(str(p))

## Step 5: Use it (run locally, not in Colab)

Pick the panel that looks best (all 3 use the same real checkpoints, none cherry-picked for content — just pick the clearest crop) and overwrite `Phase1/Dhinanjaya-Person5/paper_acm/figures/attention_drift_01_full_scale.png` with it, then fix the caption in `main.tex` (remove "λ2=1.0 map pending" — it's here now) and recompile.